<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "178"
iso_country = "Congo"
year = 2019
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 13.980*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Congo': 

Number of trade records present in dataset: 15844
Number of partner countries present in dataset: 106
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1995.3m
Total weight within the dataset: 1292.2 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,178,Congo,156,China,Import,Sea,551321,"551321 | Fabrics, woven; plai",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,199687.616,NaN,199687.616,kg,209468.0,209468.0,30771.861626,18516.0,11505.305436,354039.666837,['Container'],['Container'],1050.896146,1046.537849,1046.861283,1046.861283,1049.664577,1049.664577,1106.123262,2115.526240,1226.430872,1121.189308,1125.988622,1162.187586,1080.002266,2013.305891,1276.216522,1169.569237,1142.070188,1202.212595,1057.104738,2232.180343,1591.253732,1393.127310,1286.012411,1257.704531,1034.522901,2358.439938,1872.401213,1751.477806,1370.010822,1355.513637,1012.362850,2415.862934,2284.241752,2293.162853,1483.785612,1427.425874,989.782226,2377.187772,2299.651317,2440.893841,1829.22290,1454.575388,968.094822,2091.679438,2164.983802,2320.613390,1752.530616,1472.472078,946.515548,1863.095177,1915.093331,2245.991447,1821.261534,1484.363856,2676
1,1,178,Congo,702,Singapore,Import,Sea,551321,"551321 | Fabrics, woven; plai",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,4307.288,NaN,4307.288,kg,6735.0,6735.0,663.753081,14233.0,8843.973443,5870.214619,['Container'],['Container'],17.424561,17.352298,17.357661,17.357661,17.404141,17.404141,18.340264,35.076841,20.335045,18.590069,18.669645,19.269848,17.907160,33.381959,21.160524,19.392241,18.936288,19.933489,17.527504,37.011044,26.384052,23.098983,21.322946,20.853583,17.153082,39.104513,31.045665,29.040674,22.715696,22.475323,16.785654,40.056624,37.874257,38.022175,24.602158,23.667676,16.411252,39.415364,38.129758,40.471653,30.32974,24.117833,16.051660,34.681445,35.896880,38.477323,29.058131,24.414573,15.693861,30.891365,31.753529,37.240041,30.197735,24.611746,2676


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $552.3m (27.7%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
02 | Meat and edible meat  171686754.9   84943682.2      8.6      6.6
10 | Cereals                90805493.4  268881670.9      4.6     20.8
03 | Fish and crustaceans   76229847.3   23140051.7      3.8      1.8
19 | Preparations of cere   52160329.9   17093182.5      2.6      1.3
15 | Animal, vegetable or   52040852.9   56883238.8      2.6      4.4
11 | Products of the mill   18439419.2   35051158.5      0.9      2.7
04 | Dairy produce; birds   15621977.8    7588717.2      0.8      0.6
21 | Miscellaneous edible   14636334.2    5452680.5      0.7      0.4
22 | Beverages, spirits a   14363623.7   19856011.3      0.7      1.5
05 | Animal originated pr    9323213.8    2193813.0      0.5      0.2
16 | Meat, fish, crustace    8782399.9    1995579.1      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $429.4m (21.5%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       309948315.1  258754329.5     15.5     20.0
Food & Beverages  106788250.6   57726365.2      5.4      4.5
Fishing            12681679.7    1664026.9      0.6      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $401.7m (20.1%), ... 

                                  cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                               
OMT - Meat products n.e.c.     139768032.5  70471012.7      7.0      5.5
OFD - Food products n.e.c.     103107162.3  44246832.9      5.2      3.4
VOL - Vegetable oils and fats   54001476.4  61378396.5      2.7      4.7
CMT - Bovine meat prods         38339240.6  15943903.6      1.9      1.2
PCR - Processed rice            32338232.2  56171509.3      1.6      4.3
MIL - Dairy products            14657310.8   6717160.9      0.7      0.5
OAP - Animal products n.e.c.     9389392.3   2213568.0      0.5      0.2
V_F - Vegetables, fruit, nuts    8247315.9  12572926.2      0.4      1.0
GRO - Cereal grains n.e.c.        874614.4   3537814.2      0.0      0.3
OCR - Crops n.e.c.                640453.8    161411.3      0.0      0.0
SGR - Sugar                       287868.5    630536.7    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 27.7% 
GTAP 20.1% 
EORA 21.5%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46356/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,178,Congo,156,China,Import,Sea,551321,"551321 | Fabrics, woven; plai",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,199687.616,NaN,199687.616,kg,209468.0,209468.0,30771.861626,18516.0,11505.305436,354039.666837,['Container'],['Container'],1050.896146,1046.537849,1046.861283,1046.861283,1049.664577,1049.664577,1106.123262,2115.526240,1226.430872,1121.189308,1125.988622,1162.187586,1080.002266,2013.305891,1276.216522,1169.569237,1142.070188,1202.212595,1057.104738,2232.180343,1591.253732,1393.127310,1286.012411,1257.704531,1034.522901,2358.439938,1872.401213,1751.477806,1370.010822,1355.513637,1012.362850,2415.862934,2284.241752,2293.162853,1483.785612,1427.425874,989.782226,2377.187772,2299.651317,2440.893841,1829.22290,1454.575388,968.094822,2091.679438,2164.983802,2320.613390,1752.530616,1472.472078,946.515548,1863.095177,1915.093331,2245.991447,1821.261534,1484.363856,2676,NaN
1,1,178,Congo,702,Singapore,Import,Sea,551321,"551321 | Fabrics, woven; plai",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,4307.288,NaN,4307.288,kg,6735.0,6735.0,663.753081,14233.0,8843.973443,5870.214619,['Container'],['Container'],17.424561,17.352298,17.357661,17.357661,17.404141,17.404141,18.340264,35.076841,20.335045,18.590069,18.669645,19.269848,17.907160,33.381959,21.160524,19.392241,18.936288,19.933489,17.527504,37.011044,26.384052,23.098983,21.322946,20.853583,17.153082,39.104513,31.045665,29.040674,22.715696,22.475323,16.785654,40.056624,37.874257,38.022175,24.602158,23.667676,16.411252,39.415364,38.129758,40.471653,30.32974,24.117833,16.051660,34.681445,35.896880,38.477323,29.058131,24.414573,15.693861,30.891365,31.753529,37.240041,30.197735,24.611746,2676,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $340.9m of $1995.3m (17.1%)
Volume: 417.2 of 1292.2 million kilograms (32.3%)
Value-to-Weight: 0.82 from 1.54 US$ per kg (+-47.1%)
Average distance to market (volume-weighted): 6599.6 from 5745.9 nm (+14.9%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0        Meat and Prep    122  133672420.2   69063038.8      6.7      5.3
1    Processed Cereals    408  112490885.9  107380985.8      5.6      8.3
2  Unprocessed Cereals     28   58452174.8  212683956.5      2.9     16.5
3                Dairy     97   14851706.7    7173854.4      0.7      0.6
4           Vegetables    256    9346507.1   13033310.3      0.5      1.0
5        Fats and Oils     67    7661243.9    4612994.4      0.4      0.4
6      Sugar and Honey     54    2859190.2    1638340.5      0.1      0.1
7               Fruits    101    1574807.1    1609622.8      0.1      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,13288812.8,25361765.0,14726297.9,13469758.6,13527201.4,13960223.8,12166717.2,28990197.3,27410449.6,27527836.9,17824133.8,17138003.6,11378432.0,22356741.4,22977804.5,26937270.1,21842342.6,17819861.9
1,Fats and Oils,10388458.3,19307097.3,11433250.3,10499807.2,10542021.1,10889392.1,9554758.1,22000376.2,20838323.5,21015288.4,13806989.8,13251489.9,8969639.0,17163508.6,17656087.2,20670342.0,16809819.2,13757356.8
2,Fruits,1936207.5,3683871.1,2143940.7,1962143.0,1970412.5,2033472.1,1773720.4,4211062.8,3982107.1,4000856.6,2594080.0,2493631.2,1659584.2,3251939.8,3341631.5,3915433.7,3176218.0,2592397.2
3,Meat and Prep,134857834.6,257923594.6,149525660.8,136694677.2,137279806.4,141693160.7,123426625.6,294540450.5,278493280.9,279580935.8,180902184.7,174030841.7,115398565.1,227147362.1,233486943.5,273829828.4,222047031.4,180972683.8
4,Processed Cereals,257726852.5,483599855.6,284240800.7,260785835.2,261865269.5,270327390.1,236686320.3,554810762.6,524779478.8,528248035.7,344773206.6,329811659.9,221914226.0,430065409.3,441542582.4,516279015.7,420303210.2,342510240.8
5,Sugar and Honey,1860029.8,3435545.2,2044132.7,1882446.9,1889746.2,1948752.0,1712716.9,3952737.1,3740563.2,3775630.1,2476438.3,2368612.3,1609104.8,3073238.3,3152456.4,3675250.6,2989103.7,2458510.6
6,Unprocessed Cereals,419573017.8,801591954.4,465066411.4,425245174.4,427062104.8,440796145.3,384082898.5,915631255.4,865763790.5,869276522.0,562740864.9,541189647.5,359158897.4,706332978.4,725997454.9,851294206.2,690466524.4,562734091.0
7,Vegetables,21326250.8,40325951.3,23574684.9,21604537.5,21694220.5,22386121.0,19558371.9,46157529.8,43653135.4,43903235.6,28530970.5,27401703.8,18314210.6,35694712.4,36669731.8,42929740.9,34847791.6,28477130.3


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.81,0.10,0.01,0.02,0.05,1.13,1.03,1.03,0.38,0.33,0.74,0.78,1.05,0.70,0.43
1,Fats and Oils,1.16,0.14,0.01,0.02,0.07,1.62,1.47,1.50,0.56,0.48,1.07,1.13,1.53,1.02,0.62
2,Fruits,1.11,0.13,0.02,0.02,0.06,1.55,1.40,1.41,0.52,0.46,1.01,1.07,1.43,0.96,0.59
3,Meat and Prep,0.92,0.11,0.01,0.02,0.05,1.28,1.16,1.17,0.43,0.38,0.84,0.88,1.19,0.80,0.49
4,Processed Cereals,2.01,0.24,0.03,0.04,0.11,2.83,2.56,2.59,0.96,0.83,1.85,1.95,2.62,1.76,1.07
5,Sugar and Honey,0.55,0.06,0.01,0.01,0.03,0.78,0.71,0.72,0.27,0.23,0.51,0.54,0.72,0.48,0.30
6,Unprocessed Cereals,6.54,0.78,0.10,0.13,0.36,9.09,8.24,8.30,3.06,2.69,5.94,6.28,8.42,5.67,3.48
7,Vegetables,2.03,0.24,0.03,0.04,0.11,2.85,2.58,2.60,0.96,0.84,1.86,1.96,2.63,1.77,1.09
